# setup

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from dataclasses import replace
from pathlib import Path
from typing import Callable, Sequence
import csv as csv_module

import jax.numpy as jnp
import numpy as np

from compressible import (
    ClippingConfig,
    EquationManager,
    Mesh,
    NumericsConfig,
    build_boundary_arrays_1d,
    build_boundary_arrays_1d_periodic,
    compute_U_from_primitives,
    compute_cfl_dt as compute_cfl_dt_unified,
    extract_primitives_from_U,
    run_scan,
)
from compressible import solver as solver_module
from compressible_core import chemistry_utils, energy_models

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = "plotly_white"


def make_piecewise_fn(
    x0: float, left: jnp.ndarray | float, right: jnp.ndarray | float, width: float = 0.0
) -> Callable[[jnp.ndarray], jnp.ndarray]:
    left = jnp.asarray(left)
    right = jnp.asarray(right)

    def fn(x: jnp.ndarray) -> jnp.ndarray:
        cond = x < x0
        if left.ndim > 0 or right.ndim > 0:
            cond = cond[..., None]
        if width <= 0.0:
            return jnp.where(cond, left, right)
        blend = 0.5 * (1.0 + jnp.tanh((x - x0) / width))
        if left.ndim > 0 or right.ndim > 0:
            blend = blend[..., None]
        return left * (1.0 - blend) + right * blend

    return fn


def build_grid(n_cells: int, length: float) -> tuple[jnp.ndarray, float]:
    dx = length / n_cells
    x = jnp.linspace(0.5 * dx, length - 0.5 * dx, n_cells)
    return x, dx


def load_species_table(
    species_names: Sequence[str],
    general_data_path: str,
    energy_data_path: str,
) -> chemistry_utils.SpeciesTable:
    energy_cfg = energy_models.EnergyModelConfig(
        model="bird",
        include_electronic=False,
        data_path=str(energy_data_path),
    )
    return chemistry_utils.load_species_table(
        species_names=species_names,
        general_data_path=str(general_data_path),
        energy_model_config=energy_cfg,
    )


def normalize_mole_fractions(Y: jnp.ndarray) -> jnp.ndarray:
    Y = jnp.asarray(Y)
    if Y.ndim == 1:
        Y = Y[:, None]
    return Y / jnp.clip(jnp.sum(Y, axis=1, keepdims=True), 1e-14, None)


def build_initial_state(
    x: jnp.ndarray,
    rho_fn: Callable,
    u_fn: Callable,
    Y_fn: Callable,
    T_fn: Callable,
    Tv_fn: Callable | None = None,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    rho = rho_fn(x)
    u = u_fn(x)
    Y = normalize_mole_fractions(jnp.asarray(Y_fn(x)))
    T = T_fn(x)
    Tv = T if Tv_fn is None else Tv_fn(x)
    return rho, u, T, Tv, Y


def compute_cfl_dt(
    U_init: jnp.ndarray,
    mesh: Mesh,
    eq_manager: EquationManager,
    cfl: float = 0.45,
) -> float:
    eq_manager_cfl = replace(
        eq_manager,
        numerics_config=replace(eq_manager.numerics_config, cfl=cfl),
    )
    return compute_cfl_dt_unified(U_init, mesh, eq_manager_cfl)


def load_toro_csv(path: str) -> dict:
    result = {
        k: {"x": [], "y": []}
        for k in ("density", "velocity", "pressure", "internal_energy")
    }
    keys = list(result.keys())
    with open(path) as f:
        reader = csv_module.reader(f)
        next(reader)
        next(reader)
        for row in reader:
            row = row + [""] * (8 - len(row))
            for i, key in enumerate(keys):
                x_val, y_val = row[2 * i].strip(), row[2 * i + 1].strip()
                if x_val and y_val:
                    result[key]["x"].append(float(x_val))
                    result[key]["y"].append(float(y_val))
    return result

# run

In [2]:
print("=" * 80)
print("Shock Tube (Toro / Sod setup)")
print("=" * 80)

# --- user input start ---
tube_length = 1.0
n_cells = 500
x0 = 0.5 * tube_length

species_names = ("N2",)

T_L = 300.0  # K
T_R = 240.0  # K
rho_L = 1.0  # kg/m3
rho_R = 0.125  # kg/m3
u_L = 0.0
u_R = 0.0
Y_L = jnp.array([1.0])
Y_R = jnp.array([1.0])

boundary_condition = "outflow"
CFL = 0.45

flux_scheme = "exact_riemann"
spatial_scheme = "muscl"
integrator_scheme = "rk2"
# --- user input end ---

x, dx = build_grid(n_cells, tube_length)
x_nodes = jnp.linspace(0.0, tube_length, n_cells + 1)
mesh = Mesh.from_1d_grid(x_nodes, periodic=boundary_condition == "periodic")

data_dir = Path("/home/hhoechter/tum/jaxfluids_internship/data")

species_table = load_species_table(
    species_names,
    general_data_path=str(data_dir / "species.json"),
    energy_data_path=str(data_dir / "air_5_bird_energy.json"),
)

numerics_config = NumericsConfig(
    dt=1e-6,
    cfl=CFL,
    dt_mode="fixed",
    integrator_scheme=integrator_scheme,
    spatial_scheme=spatial_scheme,
    flux_scheme=flux_scheme,
    clipping=ClippingConfig(),
)

if boundary_condition == "periodic":
    boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species_table.n_species)
else:
    boundary_arrays = build_boundary_arrays_1d(
        mesh,
        bc_left=boundary_condition,
        bc_right=boundary_condition,
        n_species=species_table.n_species,
    )

eq_manager = EquationManager(
    species=species_table,
    collision_integrals=None,
    reactions=None,
    numerics_config=numerics_config,
    transport_model=None,
    casseau_transport=None,
    boundary_arrays=boundary_arrays,
)

rho_fn = make_piecewise_fn(x0, rho_L, rho_R)
u_fn = make_piecewise_fn(x0, u_L, u_R)
Y_fn = make_piecewise_fn(x0, Y_L, Y_R)
T_fn = make_piecewise_fn(x0, T_L, T_R)

rho, u, T, Tv, Y = build_initial_state(x, rho_fn, u_fn, Y_fn, T_fn)

U_init = compute_U_from_primitives(
    Y_s=Y,
    rho=rho,
    u=u,
    v=jnp.zeros_like(u),
    T_tr=T,
    T_V=Tv,
    equation_manager=eq_manager,
)

dt = compute_cfl_dt(U_init, mesh, eq_manager, cfl=CFL) / 2
eq_manager = replace(eq_manager, numerics_config=replace(numerics_config, dt=dt))

prim_0 = extract_primitives_from_U(U_init, eq_manager)
Y_0, rho_0, T_0, Tv_0, p_0 = prim_0.Y_s, prim_0.rho, prim_0.T, prim_0.Tv, prim_0.p
n_species = species_table.n_species
a_L = float(
    solver_module.compute_speed_of_sound(rho_0, p_0, Y_0, T_0, Tv_0, eq_manager)[0]
)

u_ref = float(jnp.sqrt(p_0[0] / rho_0[0]))
t_target = 0.25 * tube_length / u_ref
n_steps = max(1, int(round(t_target / dt)))
t_final = n_steps * dt

print(f"  dt                = {dt:.3e} s")
print(f"  a_L               = {a_L:.2f} m/s")
print(f"  u_ref             = {u_ref:.2f} m/s  (= sqrt(p_L / rho_L))")
print(f"  t_target          = {t_target:.3e} s  (Toro t* = 0.25)")
print(f"  t_compare         = {t_final:.3e} s  (= n_steps * dt)")
print(f"  n_steps           = {n_steps}")
print(f"  flux_scheme       = {flux_scheme}")
print(f"  spatial_scheme    = {spatial_scheme}")
print(f"  integrator_scheme = {integrator_scheme}")

U_hist, t_hist = run_scan(
    U_init=U_init,
    mesh=mesh,
    equation_manager=eq_manager,
    t_final=t_final,
    save_interval=1,
)
print(f"Done. U_hist shape: {U_hist.shape}")
print(f"Actual final time   = {float(t_hist[-1]):.3e} s")

Shock Tube (Toro / Sod setup)
  dt                = 6.370e-07 s
  a_L               = 353.24 m/s
  u_ref             = 298.47 m/s  (= sqrt(p_L / rho_L))
  t_target          = 8.376e-04 s  (Toro t* = 0.25)
  t_compare         = 8.376e-04 s  (= n_steps * dt)
  n_steps           = 1315
  flux_scheme       = exact_riemann
  spatial_scheme    = muscl
  integrator_scheme = rk2
Done. U_hist shape: (1316, 500, 5)
Actual final time   = 8.376e-04 s


# reference comparison

In [3]:
def run_scheme(flux_scheme, spatial_scheme, integrator_scheme):
    """Run the shock tube and return the snapshot closest to Toro's t* = 0.25."""
    _numerics = NumericsConfig(
        dt=dt,
        integrator_scheme=integrator_scheme,
        spatial_scheme=spatial_scheme,
        flux_scheme=flux_scheme,
        clipping=ClippingConfig(),
    )
    _eq = replace(eq_manager, numerics_config=_numerics)
    _U_hist, _t_hist = run_scan(
        U_init=U_init,
        mesh=mesh,
        equation_manager=_eq,
        t_final=t_final,
        save_interval=1,
    )
    _idx = int(jnp.argmin(jnp.abs(_t_hist - t_target)))
    _prim = extract_primitives_from_U(_U_hist[_idx], _eq)
    Y_f, rho_f, T_f, Tv_f, p_f = _prim.Y_s, _prim.rho, _prim.T, _prim.Tv, _prim.p
    u_f = _U_hist[_idx, :, _eq.species.n_species] / rho_f
    return (
        np.array(rho_f) / rho_L,
        np.array(u_f) / u_ref,
        np.array(p_f) / float(p_0[0]),
        float(_t_hist[_idx]),
    )


print("Running HLLC + MUSCL + RK2 ...")
rho_hllc, u_hllc, p_hllc, t_hllc = run_scheme("exact_riemann", "muscl", "rk2")
print(f"Comparing against Toro target t* = 0.25 at physical time {t_target:.6e} s")
print(f"Using HLLC snapshot at                          {t_hllc:.6e} s")

# Load Toro reference
toro_dir = Path(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/shock_tube_1d_toro"
)
toro = load_toro_csv(str(toro_dir / "toro_riemann_shocktube.csv"))

# --- plot ---
hllc_kw = dict(mode="lines", line=dict(color="black", width=2))
exact_kw = dict(mode="lines", line=dict(color="firebrick", width=2, dash="dash"))
ref_kw = dict(mode="markers", marker=dict(size=10, symbol="x", color="black"))

fig = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.10,
)

quantities = [
    ("density", rho_hllc, None, 1),
    ("velocity", u_hllc, None, 2),
    ("pressure", p_hllc, None, 3),
]

for toro_key, sim_hllc, sim_exact, col in quantities:
    show = col == 1
    fig.add_trace(
        go.Scatter(
            x=np.array(x),
            y=sim_hllc,
            name="HLLC + MUSCL + RK2",
            showlegend=show,
            **hllc_kw,
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=toro[toro_key]["x"],
            y=toro[toro_key]["y"],
            name="Toro reference",
            showlegend=show,
            **ref_kw,
        ),
        row=1,
        col=col,
    )

_ax_x = dict(
    title_font=dict(size=22), tickfont=dict(size=18), tickangle=0, title_text="x [m]"
)
_ax_y = dict(title_font=dict(size=22), tickfont=dict(size=18))
fig.update_layout(
    template="simple_white",
    title=None,
    width=1400,
    height=500,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=_ax_x,
    xaxis2=_ax_x,
    xaxis3=_ax_x,
    yaxis={**_ax_y, "title_text": "rho / rho_L"},
    yaxis2={**_ax_y, "title_text": "u / u_ref"},
    yaxis3={**_ax_y, "title_text": "p / p_L"},
    legend=dict(
        font=dict(size=18),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.show()
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/shock_tube_1d_toro/shock_tube_1d_toro.pdf"
)

Running HLLC + MUSCL + RK2 ...
Comparing against Toro target t* = 0.25 at physical time 8.376090e-04 s
Using HLLC snapshot at                          8.376066e-04 s


# time evolution

In [4]:
snapshot_times = jnp.array([0.0, 0.25 * t_final, 0.5 * t_final, t_final])
indices = [int(jnp.argmin(jnp.abs(t_hist - t_target))) for t_target in snapshot_times]

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=["rho [kg/m3]", "u [m/s]", "p [Pa]"],
    vertical_spacing=0.07,
)

colors = ["royalblue", "darkorange", "firebrick", "green"]

for k, idx in enumerate(indices):
    _prim_i = extract_primitives_from_U(U_hist[idx], eq_manager)
    Y_i, rho_i, T_i, Tv_i, p_i = (
        _prim_i.Y_s,
        _prim_i.rho,
        _prim_i.T,
        _prim_i.Tv,
        _prim_i.p,
    )
    u_i = U_hist[idx, :, n_species] / rho_i
    label = f"t = {float(t_hist[idx]):.2e} s"
    kw = dict(mode="lines", line=dict(color=colors[k]))
    fig.add_trace(
        go.Scatter(x=np.array(x), y=np.array(rho_i), name=label, **kw), row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=np.array(x), y=np.array(u_i), name=label, showlegend=False, **kw),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=np.array(x), y=np.array(p_i), name=label, showlegend=False, **kw),
        row=3,
        col=1,
    )

_ax_x = dict(title_font=dict(size=22), tickfont=dict(size=18), tickangle=45)
_ax_y = dict(title_font=dict(size=22), tickfont=dict(size=18))
fig.update_layout(
    title=None,
    width=1200,
    height=900,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=_ax_x,
    xaxis2=_ax_x,
    xaxis3={**_ax_x, "title_text": "x [m]"},
    yaxis=_ax_y,
    yaxis2=_ax_y,
    yaxis3=_ax_y,
    legend=dict(
        font=dict(size=18),
        x=0.5,
        y=-0.12,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.show()